# Music Annotation Project V9.0B
## Notebook 60R1: Fresh SAFE_REMOVE inference on reviewed HITL masks

**Runtime:** GPU. In Google Colab select a T4 GPU, then choose **Runtime → Run all**.

### Purpose
Operationalise the frozen V8.8 REMOVE-only policy on the real Notebook 06 → Notebook 07 reviewed HITL population.

### Page-specific runtime inputs
- Notebook 07 aligned observed page.
- Notebook 07 reviewed/corrected authoritative mask.
- Notebook 06 draft/confidence lineage for audit only.

### Frozen policy inputs
- One complete Notebook 52 V8.8A freeze/replay run.
- The exact model/checkpoint, calibrator, ordered channel schema, thresholds, connected-run rules and compatibility identity named by that run.
- Canonical V7.8 mask preparation, observable channels and baseline reconstruction.

### Non-negotiable rules
- No Notebook 45, 46 or 50 **page output** is required for a new HITL page.
- Notebook 50 may be referenced only as the recorded producer of frozen learned objects named by Notebook 52.
- No clean page is loaded.
- No training, fitting, recalibration, threshold change or policy selection occurs.
- SAFE_PROTECT is prohibited.
- `M5_REMOVE = M3 & ~SAFE_REMOVE`.
- Unsupported, uncertain, invalid and failed coordinates retain exact M3 treatment.
- No pixel outside the authoritative prepared mask may change.
- The notebook stops if the frozen bundle cannot unambiguously describe fresh-page inference. It never invents an ensemble rule or selects a convenient model file.

In [9]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow torch torchvision scikit-learn joblib matplotlib

## 1. Configuration and run identity

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='42'

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, sys, platform, gc, time, shutil, traceback, zipfile
import cv2, torch, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
N52_ROOT=ROOT/'m5_remove_policy_freeze_v8_8a'
OUTPUT_ROOT=ROOT/'v9_0b_notebook60r1_fresh_safe_remove'

NOTEBOOK52_RUN_OVERRIDE=None
PAGE_ALLOWLIST=None
EXPECTED_REVIEWED_PAGES=151
REQUIRE_EXACT_151=False
BATCH_SIZE=1024
STOP_AT_FIRST_PAGE_FAILURE=False
MAX_VISUAL_PAGES=30
SEED=42
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('Notebook 60R1 requires a GPU runtime for fresh frozen CNN inference.')
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook60r1_v9_0b'
OUT=OUTPUT_ROOT/RUN
INPUTS=OUT/'inputs'; PAGES=OUT/'pages'; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; FAILURES=OUT/'failures'; VISUAL=OUT/'visual'; UPLOAD=OUT/'upload'
for d in [OUT,INPUTS,PAGES,TABLES,AUDIT,FAILURES,VISUAL,UPLOAD]: d.mkdir(parents=True,exist_ok=False)
print('GPU:',torch.cuda.get_device_name(0))
print('Output:',OUT)

## 2. General helpers

In [11]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def require(x,msg):
    if not x: raise RuntimeError(msg)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def atomic_json(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def read_json(path): return json.loads(Path(path).read_text())

def read_gray(path):
    a=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    require(a is not None,f'Unreadable image: {path}')
    return a

def latest_complete(roots,required_any):
    hits=[]
    for root in roots:
        if not Path(root).exists(): continue
        for p in Path(root).rglob('*'):
            if not p.is_dir(): continue
            if all(any((p/name).exists() for name in choices) for choices in required_any): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def md5_file(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def binary(a): return (np.asarray(a)>0)

def reset_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
reset_seed()

## 3. Resolve the frozen Notebook 52 policy bundle

This resolver is manifest-led. It does not select an arbitrary `.pt` or `.joblib`. It first resolves one complete Notebook 52 run, then reads its policy and artifact records. All paths and hashes must be named by those records.

In [ ]:
# ============================================================
# NOTEBOOK 52 SOURCE AND FROZEN-ARTEFACT RESOLUTION
# ============================================================

N52_REQUIRED = [
    'notebook52_verdict.json',
    'policy_bundle/policy_manifest.json',
    'replay/source_fingerprint.json',
    'tables/notebook50_full_artifact_inventory.csv',
    'tables/binary_artifact_loadability.csv',
    'tables/m5_remove_golden_master_semantic_hashes.csv',
]


# ------------------------------------------------------------
# 1. Resolve one complete Notebook 52 run
# ------------------------------------------------------------

def latest_complete_n52(root, required):
    root = Path(root)

    if not root.exists():
        return None

    candidates = []

    # Notebook 52 writes timestamped run directories directly below
    # m5_remove_policy_freeze_v8_8a.
    for path in root.iterdir():
        if not path.is_dir():
            continue

        missing = [
            relative_path
            for relative_path in required
            if not (path / relative_path).exists()
        ]

        if not missing:
            candidates.append(path)

    return (
        max(candidates, key=lambda path: path.stat().st_mtime)
        if candidates
        else None
    )


N52 = (
    Path(NOTEBOOK52_RUN_OVERRIDE)
    if NOTEBOOK52_RUN_OVERRIDE
    else latest_complete_n52(N52_ROOT, N52_REQUIRED)
)

require(
    N52 is not None,
    f'No complete Notebook 52 run found below {N52_ROOT}',
)

if NOTEBOOK52_RUN_OVERRIDE:
    missing_override_files = [
        relative_path
        for relative_path in N52_REQUIRED
        if not (N52 / relative_path).exists()
    ]

    require(
        not missing_override_files,
        (
            f'Notebook 52 override is incomplete: {N52}. '
            f'Missing: {missing_override_files}'
        ),
    )


# ------------------------------------------------------------
# 2. Load and validate Notebook 52 control records
# ------------------------------------------------------------

N52_VERDICT_PATH = N52 / 'notebook52_verdict.json'
POLICY_PATH = N52 / 'policy_bundle' / 'policy_manifest.json'
REPLAY_PATH = N52 / 'replay' / 'source_fingerprint.json'
INVENTORY_PATH = (
    N52
    / 'tables'
    / 'notebook50_full_artifact_inventory.csv'
)

v52 = read_json(N52_VERDICT_PATH)
policy = read_json(POLICY_PATH)
replay52 = read_json(REPLAY_PATH)
inventory52 = pd.read_csv(INVENTORY_PATH)

require(
    v52.get('verdict')
    == 'PASS_M5_REMOVE_FREEZE_READY_FOR_OPERATIONAL_BUILDER',
    f"Unexpected Notebook 52 verdict: {v52.get('verdict')}",
)

require(
    v52.get('first_failed_gate') is None,
    (
        'Notebook 52 records a failed gate: '
        f"{v52.get('first_failed_gate')}"
    ),
)

require(
    policy.get('policy') == 'M5_REMOVE',
    f"Unexpected frozen policy: {policy.get('policy')}",
)

require(
    policy.get('policy_definition')
    == 'M5_REMOVE = M3 minus SAFE_REMOVE',
    (
        'Unexpected policy definition: '
        f"{policy.get('policy_definition')}"
    ),
)

require(
    policy.get('safe_protect_runtime_enabled') is False,
    'SAFE_PROTECT is enabled in the frozen runtime policy.',
)

require(
    policy.get('m3_fallback') is True,
    'Frozen policy does not require exact M3 fallback.',
)

require(
    policy.get('policy_id') == v52.get('policy_id'),
    (
        'Notebook 52 verdict and policy manifest '
        'have different policy IDs.'
    ),
)


# ------------------------------------------------------------
# 3. Normalize Notebook 52 inventory columns
# ------------------------------------------------------------

normalised_columns = {
    str(column).lower().strip(): column
    for column in inventory52.columns
}

path_col = next(
    (
        normalised_columns[name]
        for name in [
            'relative_path',
            'path',
            'file',
            'artifact_path',
            'artefact_path',
        ]
        if name in normalised_columns
    ),
    None,
)

hash_col = next(
    (
        normalised_columns[name]
        for name in [
            'sha256',
            'file_sha256',
            'hash',
        ]
        if name in normalised_columns
    ),
    None,
)

role_col = next(
    (
        normalised_columns[name]
        for name in [
            'role',
            'semantic_role',
            'artifact_role',
            'artefact_role',
        ]
        if name in normalised_columns
    ),
    None,
)

require(
    path_col is not None
    and hash_col is not None
    and role_col is not None,
    (
        'Notebook 52 inventory lacks a usable '
        'path/hash/role schema. '
        f'Columns: {list(inventory52.columns)}'
    ),
)


# ------------------------------------------------------------
# 4. Resolve the authoritative Notebook 50 source run
# ------------------------------------------------------------

source_run_value = (
    policy.get('source_notebook50_run')
    or policy.get('authoritative_notebook50_run')
    or policy.get('source_run')
)

require(
    source_run_value not in [None, ''],
    (
        'Notebook 52 policy manifest does not identify '
        'the authoritative Notebook 50 source run.'
    ),
)

SOURCE_NOTEBOOK50_RUN = Path(str(source_run_value))

require(
    SOURCE_NOTEBOOK50_RUN.exists()
    and SOURCE_NOTEBOOK50_RUN.is_dir(),
    (
        'Authoritative Notebook 50 run does not exist: '
        f'{SOURCE_NOTEBOOK50_RUN}'
    ),
)


# ------------------------------------------------------------
# 5. Runtime-dependency classification
# ------------------------------------------------------------

# The Notebook 50 full inventory contains provenance and review files as well
# as executable/semantic runtime artefacts. Only runtime-required objects
# should be allowed to block inference.

NON_RUNTIME_FILENAMES = {
    'executive_summary.txt',
    'readme.txt',
    'upload_manifest.csv',
    'upload_file_list.txt',
}

NON_RUNTIME_PATH_PARTS = {
    'upload',
    'uploads',
    'review',
    'visual',
    'visuals',
    'contact_sheets',
    'charts',
}

NON_RUNTIME_ROLE_TERMS = {
    'summary',
    'readme',
    'upload',
    'review',
    'visual',
    'chart',
    'report',
    'other',
}

RUNTIME_ROLE_TERMS = {
    'model',
    'checkpoint',
    'calibrator',
    'isotonic',
    'schema',
    'policy',
    'decision',
    'reason_code',
    'reason code',
    'semantic_array',
    'semantic array',
    'array_container',
    'array container',
    'predictor',
    'selected_model',
    'selected model',
    'implementation',
    'replay',
}

RUNTIME_FILENAMES = {
    'selected_models.json',
    'predictor_schema.json',
    'reason_code_schema.json',
    'experiment_identity.json',
    'notebook50_verdict.json',
}


def classify_inventory_entry(raw_value, role_value):
    raw_text = str(raw_value).replace('\\', '/')
    raw_path = Path(raw_text)
    filename = raw_path.name.lower()
    path_parts = {
        part.lower()
        for part in raw_path.parts
    }
    role = str(role_value).strip().lower()

    if filename in NON_RUNTIME_FILENAMES:
        return False, 'non_runtime_filename'

    if path_parts & NON_RUNTIME_PATH_PARTS:
        return False, 'non_runtime_directory'

    if any(term in role for term in NON_RUNTIME_ROLE_TERMS):
        return False, 'non_runtime_role'

    if filename in RUNTIME_FILENAMES:
        return True, 'runtime_filename'

    if any(term in role for term in RUNTIME_ROLE_TERMS):
        return True, 'runtime_role'

    # Unknown inventory records remain provenance records.
    # They are reported, but do not block runtime until explicitly designated.
    return False, 'provenance_only_unclassified'


inventory_classification = []

for _, inventory_row in inventory52.iterrows():
    runtime_required, classification_reason = (
        classify_inventory_entry(
            inventory_row[path_col],
            inventory_row[role_col],
        )
    )

    inventory_classification.append({
        'inventory_path': str(inventory_row[path_col]),
        'role': str(inventory_row[role_col]),
        'runtime_required': runtime_required,
        'classification_reason': classification_reason,
    })

classification_df = pd.DataFrame(inventory_classification)

inventory52 = inventory52.merge(
    classification_df,
    left_on=[path_col, role_col],
    right_on=['inventory_path', 'role'],
    how='left',
    validate='many_to_one',
)

classification_df.to_csv(
    AUDIT / 'notebook50_inventory_runtime_classification.csv',
    index=False,
)


# ------------------------------------------------------------
# 6. Content-addressed path resolution
# ------------------------------------------------------------

def unique_existing_files(paths):
    result = []
    seen = set()

    for candidate in paths:
        candidate = Path(candidate)

        if not candidate.exists() or not candidate.is_file():
            continue

        try:
            identity = str(candidate.resolve())
        except Exception:
            identity = str(candidate)

        if identity not in seen:
            seen.add(identity)
            result.append(candidate)

    return result


def suffix_candidates(root, raw):
    """
    Build candidate paths while preserving useful subdirectory suffixes.

    Example:
        raw = upload/executive_summary.txt

    Candidate locations include:
        root/upload/executive_summary.txt
        root/executive_summary.txt

    This function does not recursively search by basename.
    """
    raw = Path(str(raw))
    candidates = []

    if raw.is_absolute():
        candidates.append(raw)
        return candidates

    candidates.append(root / raw)

    if len(raw.parts) > 1:
        candidates.append(root / raw.name)

    return candidates


def resolve_inventory_path(
    raw_value,
    expected_sha256,
    runtime_required,
):
    """
    Resolve a Notebook 52 inventory entry by SHA-256 content identity.

    Multiple byte-identical copies are legitimate. The selected copy follows
    this deterministic preference:

      1. Notebook 52 policy_bundle;
      2. authoritative Notebook 50 source run;
      3. another unique hash-matching Notebook 52 copy.

    Non-runtime provenance files are permitted to be unresolved. They are
    recorded as warnings rather than runtime failures.
    """

    raw = Path(str(raw_value))
    expected_sha256 = str(expected_sha256).strip().lower()

    require(
        len(expected_sha256) == 64
        and all(
            character in '0123456789abcdef'
            for character in expected_sha256
        ),
        f'Invalid expected SHA-256 for {raw}: {expected_sha256}',
    )

    candidates = []

    if raw.is_absolute():
        candidates.append(raw)

    else:
        candidates.extend(
            suffix_candidates(N52, raw)
        )
        candidates.extend(
            suffix_candidates(N52 / 'policy_bundle', raw)
        )
        candidates.extend(
            suffix_candidates(N52 / 'replay', raw)
        )
        candidates.extend(
            suffix_candidates(N52 / 'tables', raw)
        )
        candidates.extend(
            suffix_candidates(SOURCE_NOTEBOOK50_RUN, raw)
        )
        candidates.extend(
            suffix_candidates(
                SOURCE_NOTEBOOK50_RUN / 'inputs',
                raw,
            )
        )
        candidates.extend(
            suffix_candidates(
                SOURCE_NOTEBOOK50_RUN / 'models',
                raw,
            )
        )
        candidates.extend(
            suffix_candidates(
                SOURCE_NOTEBOOK50_RUN / 'calibrators',
                raw,
            )
        )
        candidates.extend(
            suffix_candidates(
                SOURCE_NOTEBOOK50_RUN / 'tables',
                raw,
            )
        )
        candidates.extend(
            suffix_candidates(
                SOURCE_NOTEBOOK50_RUN / 'upload',
                raw,
            )
        )

    existing = unique_existing_files(candidates)

    inspected = []

    for candidate in existing:
        actual_sha256 = sha256_file(candidate).lower()

        inspected.append({
            'path': str(candidate),
            'sha256': actual_sha256,
            'matches_expected': (
                actual_sha256 == expected_sha256
            ),
        })

    matching = [
        Path(entry['path'])
        for entry in inspected
        if entry['matches_expected']
    ]

    if not matching:
        return {
            'status': (
                'FAIL_RUNTIME_REQUIRED'
                if runtime_required
                else 'WARN_PROVENANCE_UNRESOLVED'
            ),
            'path': None,
            'resolution_method': None,
            'matching_copies': [],
            'inspected_candidates': inspected,
            'error': (
                f'No permitted copy of {raw} matches '
                f'expected SHA-256 {expected_sha256}.'
            ),
        }

    policy_bundle_root = N52 / 'policy_bundle'

    policy_bundle_matches = [
        path
        for path in matching
        if (
            path == policy_bundle_root
            or policy_bundle_root in path.parents
        )
    ]

    if policy_bundle_matches:
        # If duplicate matching paths remain, choose deterministically.
        selected = sorted(
            policy_bundle_matches,
            key=lambda path: (
                len(path.parts),
                str(path),
            ),
        )[0]

        method = 'notebook52_policy_bundle_hash_match'

    else:
        notebook50_matches = [
            path
            for path in matching
            if (
                path == SOURCE_NOTEBOOK50_RUN
                or SOURCE_NOTEBOOK50_RUN in path.parents
            )
        ]

        if notebook50_matches:
            selected = sorted(
                notebook50_matches,
                key=lambda path: (
                    len(path.parts),
                    str(path),
                ),
            )[0]

            method = 'authoritative_notebook50_hash_match'

        else:
            selected = sorted(
                matching,
                key=lambda path: (
                    len(path.parts),
                    str(path),
                ),
            )[0]

            method = 'other_notebook52_hash_match'

    return {
        'status': 'RESOLVED',
        'path': selected,
        'resolution_method': method,
        'matching_copies': matching,
        'inspected_candidates': inspected,
        'error': None,
    }


# ------------------------------------------------------------
# 7. Resolve the complete inventory
# ------------------------------------------------------------

resolved_rows = []
runtime_failure_rows = []
provenance_warning_rows = []
replica_rows = []

for _, row in inventory52.iterrows():

    role = str(row[role_col])
    inventory_path = str(row[path_col])
    expected_hash = str(row[hash_col]).strip().lower()
    runtime_required = bool(row['runtime_required'])
    classification_reason = str(row['classification_reason'])

    try:
        resolution = resolve_inventory_path(
            raw_value=inventory_path,
            expected_sha256=expected_hash,
            runtime_required=runtime_required,
        )

        if resolution['status'] == 'RESOLVED':
            resolved_path = Path(resolution['path'])
            actual_hash = sha256_file(resolved_path).lower()

            require(
                actual_hash == expected_hash,
                (
                    'Hash changed after artefact resolution: '
                    f'{resolved_path}'
                ),
            )

            resolved_rows.append({
                'role': role,
                'inventory_path': inventory_path,
                'path': str(resolved_path),
                'sha256': actual_hash,
                'bytes': int(resolved_path.stat().st_size),
                'runtime_required': runtime_required,
                'classification_reason': classification_reason,
                'resolution_method': (
                    resolution['resolution_method']
                ),
                'matching_copy_count': len(
                    resolution['matching_copies']
                ),
            })

            for copy_path in resolution['matching_copies']:
                replica_rows.append({
                    'role': role,
                    'inventory_path': inventory_path,
                    'runtime_required': runtime_required,
                    'expected_sha256': expected_hash,
                    'copy_path': str(copy_path),
                    'selected': (
                        Path(copy_path) == resolved_path
                    ),
                    'copy_sha256': sha256_file(copy_path),
                })

        else:
            failure_record = {
                'role': role,
                'inventory_path': inventory_path,
                'expected_sha256': expected_hash,
                'runtime_required': runtime_required,
                'classification_reason': classification_reason,
                'status': resolution['status'],
                'error': resolution['error'],
                'inspected_candidates': json.dumps(
                    resolution['inspected_candidates'],
                    sort_keys=True,
                ),
            }

            if runtime_required:
                runtime_failure_rows.append(failure_record)
            else:
                provenance_warning_rows.append(failure_record)

    except Exception as error:
        failure_record = {
            'role': role,
            'inventory_path': inventory_path,
            'expected_sha256': expected_hash,
            'runtime_required': runtime_required,
            'classification_reason': classification_reason,
            'status': (
                'FAIL_RUNTIME_REQUIRED'
                if runtime_required
                else 'WARN_PROVENANCE_UNRESOLVED'
            ),
            'error': str(error),
            'inspected_candidates': None,
        }

        if runtime_required:
            runtime_failure_rows.append(failure_record)
        else:
            provenance_warning_rows.append(failure_record)


# ------------------------------------------------------------
# 8. Build reports only AFTER every inventory row is processed
# ------------------------------------------------------------

resolved_columns = [
    'role',
    'inventory_path',
    'path',
    'sha256',
    'bytes',
    'runtime_required',
    'classification_reason',
    'resolution_method',
    'matching_copy_count',
]

failure_columns = [
    'role',
    'inventory_path',
    'expected_sha256',
    'runtime_required',
    'classification_reason',
    'status',
    'error',
    'inspected_candidates',
]

replica_columns = [
    'role',
    'inventory_path',
    'runtime_required',
    'expected_sha256',
    'copy_path',
    'selected',
    'copy_sha256',
]

resolved_df = pd.DataFrame(
    resolved_rows,
    columns=resolved_columns,
)

runtime_failure_df = pd.DataFrame(
    runtime_failure_rows,
    columns=failure_columns,
)

provenance_warning_df = pd.DataFrame(
    provenance_warning_rows,
    columns=failure_columns,
)

replica_df = pd.DataFrame(
    replica_rows,
    columns=replica_columns,
)

resolved_df.to_csv(
    AUDIT / 'resolved_frozen_artifacts.csv',
    index=False,
)

runtime_failure_df.to_csv(
    AUDIT / 'frozen_runtime_artifact_failures.csv',
    index=False,
)

provenance_warning_df.to_csv(
    AUDIT / 'frozen_provenance_artifact_warnings.csv',
    index=False,
)

replica_df.to_csv(
    AUDIT / 'frozen_artifact_replica_inventory.csv',
    index=False,
)


# ------------------------------------------------------------
# 9. Explicit required runtime controls
# ------------------------------------------------------------

resolved_names = {
    Path(path).name.lower()
    for path in resolved_df.loc[
        resolved_df['runtime_required'],
        'path',
    ].astype(str)
}

required_control_names = {
    'selected_models.json',
    'predictor_schema.json',
    'reason_code_schema.json',
    'experiment_identity.json',
    'notebook50_verdict.json',
}

missing_required_controls = sorted(
    required_control_names - resolved_names
)

require(
    not missing_required_controls,
    (
        'Required frozen Notebook 50 control artefacts '
        f'were not resolved: {missing_required_controls}'
    ),
)

require(
    runtime_failure_df.empty,
    (
        'One or more runtime-required frozen artefacts '
        'failed content-addressed resolution. See '
        f'{AUDIT / "frozen_runtime_artifact_failures.csv"}'
    ),
)


# ------------------------------------------------------------
# 10. Write source identity
# ------------------------------------------------------------

source_record = {
    'notebook52_run': str(N52),
    'notebook52_verdict_path': str(N52_VERDICT_PATH),
    'notebook52_verdict_sha256': (
        sha256_file(N52_VERDICT_PATH)
    ),
    'policy_path': str(POLICY_PATH),
    'policy_sha256': sha256_file(POLICY_PATH),
    'inventory_path': str(INVENTORY_PATH),
    'inventory_sha256': sha256_file(INVENTORY_PATH),
    'replay_path': str(REPLAY_PATH),
    'replay_sha256': sha256_file(REPLAY_PATH),
    'source_notebook50_run': str(SOURCE_NOTEBOOK50_RUN),
    'policy_id': policy.get('policy_id'),
    'compatibility_id': policy.get('compatibility_id'),
    'inventory_rows': int(len(inventory52)),
    'resolved_artifacts': int(len(resolved_df)),
    'runtime_required_resolved': int(
        resolved_df['runtime_required'].sum()
        if len(resolved_df)
        else 0
    ),
    'runtime_resolution_failures': int(
        len(runtime_failure_df)
    ),
    'provenance_resolution_warnings': int(
        len(provenance_warning_df)
    ),
    'physical_matching_copies': int(len(replica_df)),
}

atomic_json(
    AUDIT / 'notebook52_source.json',
    source_record,
)


# ------------------------------------------------------------
# 11. Concise result display
# ------------------------------------------------------------

print('Notebook 52:', N52)
print('Policy ID:', policy.get('policy_id'))
print('Compatibility ID:', policy.get('compatibility_id'))
print('Authoritative Notebook 50:', SOURCE_NOTEBOOK50_RUN)
print('Inventory rows:', len(inventory52))
print('Resolved artefacts:', len(resolved_df))
print('Runtime failures:', len(runtime_failure_df))
print('Provenance warnings:', len(provenance_warning_df))
print('Physical matching copies:', len(replica_df))

if len(provenance_warning_df):
    print(
        '\nNon-blocking provenance warnings were written to:',
        AUDIT / 'frozen_provenance_artifact_warnings.csv',
    )

display(
    resolved_df[
        [
            'role',
            'inventory_path',
            'path',
            'runtime_required',
            'resolution_method',
            'matching_copy_count',
            'sha256',
        ]
    ]
)

if len(provenance_warning_df):
    display(
        provenance_warning_df[
            [
                'role',
                'inventory_path',
                'classification_reason',
                'status',
                'error',
            ]
        ]
    )

## 4. Interpret the exact frozen fresh-inference contract

In [ ]:
# ============================================================
# CODE CELL 4
# Resolve the exact Notebook 50 frozen control records
# ============================================================
#
# PROVENANCE:
# Notebook 52 freezes and validates the authoritative Notebook 50 run.
# Notebook 50's exact control records, not broad inventory role strings,
# establish the predictor schema and selected learned objects.
#
# Notebook 52 inventory roles are deliberately broad:
#   model
#   calibrator_or_serialised_object
#   schema
#   manifest_or_identity
#   review_table
#   json_record
# etc.
#
# Therefore this cell resolves the known authoritative control records
# by exact relative path and verifies them against the Notebook 52
# content-addressed inventory.
# ============================================================


# ------------------------------------------------------------
# 1. Exact authoritative Notebook 50 control records
# ------------------------------------------------------------

SELECTED_MODELS_PATH = (
    SOURCE_NOTEBOOK50_RUN
    / 'selected_models.json'
)

PREDICTOR_SCHEMA_PATH = (
    SOURCE_NOTEBOOK50_RUN
    / 'inputs'
    / 'predictor_schema.json'
)

REASON_CODE_SCHEMA_PATH = (
    SOURCE_NOTEBOOK50_RUN
    / 'reason_code_schema.json'
)

N50_IDENTITY_PATH = (
    SOURCE_NOTEBOOK50_RUN
    / 'experiment_identity.json'
)

N50_VERDICT_PATH = (
    SOURCE_NOTEBOOK50_RUN
    / 'notebook50_verdict.json'
)


required_control_paths = {
    'selected_models':
        SELECTED_MODELS_PATH,

    'predictor_schema':
        PREDICTOR_SCHEMA_PATH,

    'reason_code_schema':
        REASON_CODE_SCHEMA_PATH,

    'experiment_identity':
        N50_IDENTITY_PATH,

    'notebook50_verdict':
        N50_VERDICT_PATH,
}


for name, path in required_control_paths.items():

    require(
        path.exists()
        and path.is_file(),
        (
            f'Missing authoritative Notebook 50 '
            f'control record {name}: {path}'
        ),
    )


# ------------------------------------------------------------
# 2. Build a relative-path lookup from Notebook 52 inventory
# ------------------------------------------------------------

inventory_lookup = {}

for _, row in inventory52.iterrows():

    relative_path = (
        str(row[path_col])
        .replace('\\', '/')
        .lstrip('./')
    )

    inventory_lookup.setdefault(
        relative_path,
        []
    ).append(row)


def inventory_hash_for(relative_path):
    """
    Return the exact Notebook 52 recorded SHA-256
    for one Notebook 50 relative path.

    Filename-only matching is prohibited.
    """

    relative_path = (
        str(relative_path)
        .replace('\\', '/')
        .lstrip('./')
    )

    matches = inventory_lookup.get(
        relative_path,
        [],
    )

    require(
        len(matches) == 1,
        (
            f'Expected exactly one Notebook 52 '
            f'inventory record for {relative_path}; '
            f'found {len(matches)}.'
        ),
    )

    expected = str(
        matches[0][hash_col]
    ).strip().lower()

    require(
        len(expected) == 64
        and all(
            character
            in '0123456789abcdef'
            for character in expected
        ),
        (
            f'Notebook 52 recorded invalid SHA-256 '
            f'for {relative_path}: {expected!r}'
        ),
    )

    return expected


# ------------------------------------------------------------
# 3. Verify exact Notebook 50 controls against Notebook 52
# ------------------------------------------------------------

control_identity_rows = []

for name, path in required_control_paths.items():

    relative_path = str(
        path.relative_to(
            SOURCE_NOTEBOOK50_RUN
        )
    ).replace('\\', '/')

    expected_sha256 = inventory_hash_for(
        relative_path
    )

    actual_sha256 = sha256_file(
        path
    ).lower()

    require(
        actual_sha256
        == expected_sha256,
        (
            f'Notebook 50 control-record hash mismatch '
            f'for {relative_path}: '
            f'expected={expected_sha256}, '
            f'actual={actual_sha256}'
        ),
    )

    control_identity_rows.append({
        'control':
            name,

        'relative_path':
            relative_path,

        'path':
            str(path),

        'sha256':
            actual_sha256,

        'notebook52_verified':
            True,
    })


control_identity_df = pd.DataFrame(
    control_identity_rows
)

control_identity_df.to_csv(
    AUDIT
    / 'notebook50_control_record_identity.csv',
    index=False,
)


# ------------------------------------------------------------
# 4. Load exact source records
# ------------------------------------------------------------

selected_models = read_json(
    SELECTED_MODELS_PATH
)

schema = read_json(
    PREDICTOR_SCHEMA_PATH
)

reason_code_schema = read_json(
    REASON_CODE_SCHEMA_PATH
)

n50_identity = read_json(
    N50_IDENTITY_PATH
)

n50_verdict = read_json(
    N50_VERDICT_PATH
)


require(
    isinstance(selected_models, dict)
    and len(selected_models) > 0,
    'Notebook 50 selected_models.json is empty.',
)

require(
    isinstance(schema, dict),
    'Notebook 50 predictor schema is not a JSON object.',
)

require(
    isinstance(reason_code_schema, dict),
    'Notebook 50 reason-code schema is not a JSON object.',
)


# ------------------------------------------------------------
# 5. Resolve exact ordered predictor schema
# ------------------------------------------------------------

PREDICTORS = (
    schema.get('ordered_predictors')
    or schema.get('predictors')
    or schema.get('ordered_features')
)


require(
    isinstance(PREDICTORS, list)
    and len(PREDICTORS) > 0,
    (
        'Notebook 50 predictor_schema.json '
        'contains no non-empty ordered predictor list.'
    ),
)


PREDICTORS = [
    str(value)
    for value in PREDICTORS
]


require(
    len(PREDICTORS)
    == len(set(PREDICTORS)),
    (
        'Frozen predictor schema contains '
        'duplicate predictor names.'
    ),
)


# ------------------------------------------------------------
# 6. Inspect selected_models.json without guessing its schema
# ------------------------------------------------------------

def flatten_json_paths(
    value,
    prefix='',
):
    """
    Produce diagnostic leaf paths from selected_models.json.

    This does NOT assign semantics.
    It gives us an auditable view of what Notebook 50
    actually recorded.
    """

    rows = []

    if isinstance(value, dict):

        for key, child in value.items():

            child_prefix = (
                f'{prefix}.{key}'
                if prefix
                else str(key)
            )

            rows.extend(
                flatten_json_paths(
                    child,
                    child_prefix,
                )
            )

    elif isinstance(value, list):

        for index, child in enumerate(value):

            rows.extend(
                flatten_json_paths(
                    child,
                    f'{prefix}[{index}]',
                )
            )

    else:

        rows.append({
            'json_path':
                prefix,

            'value':
                value,

            'value_type':
                type(value).__name__,
        })

    return rows


selected_models_fields = pd.DataFrame(
    flatten_json_paths(
        selected_models
    )
)


selected_models_fields.to_csv(
    AUDIT
    / 'selected_models_schema_inventory.csv',
    index=False,
)


# ------------------------------------------------------------
# 7. Resolve any file references contained in selected_models
# ------------------------------------------------------------

KNOWN_BINARY_SUFFIXES = {
    '.pt',
    '.pth',
    '.joblib',
    '.pkl',
    '.pickle',
    '.npz',
    '.parquet',
}


def possible_file_reference(value):

    if not isinstance(value, str):
        return False

    value = value.strip()

    if not value:
        return False

    return (
        Path(value).suffix.lower()
        in KNOWN_BINARY_SUFFIXES
    )


referenced_files = []

for row in flatten_json_paths(
    selected_models
):

    value = row['value']

    if not possible_file_reference(
        value
    ):
        continue

    raw = Path(value)

    candidates = []

    if raw.is_absolute():

        candidates.append(raw)

    else:

        candidates.extend([
            SOURCE_NOTEBOOK50_RUN
            / raw,

            SOURCE_NOTEBOOK50_RUN
            / 'models'
            / raw.name,

            SOURCE_NOTEBOOK50_RUN
            / 'calibrators'
            / raw.name,

            SOURCE_NOTEBOOK50_RUN
            / 'inputs'
            / raw.name,
        ])

    existing = []

    seen = set()

    for candidate in candidates:

        if (
            candidate.exists()
            and candidate.is_file()
        ):

            try:
                identity = str(
                    candidate.resolve()
                )
            except Exception:
                identity = str(
                    candidate
                )

            if identity not in seen:

                seen.add(identity)
                existing.append(
                    candidate
                )

    require(
        len(existing) == 1,
        (
            f'selected_models.json field '
            f'{row["json_path"]} references '
            f'{value!r}, but it resolves to '
            f'{len(existing)} files: '
            f'{existing}'
        ),
    )

    resolved_path = existing[0]

    relative_path = str(
        resolved_path.relative_to(
            SOURCE_NOTEBOOK50_RUN
        )
    ).replace('\\', '/')

    expected_sha256 = (
        inventory_hash_for(
            relative_path
        )
    )

    actual_sha256 = (
        sha256_file(
            resolved_path
        ).lower()
    )

    require(
        actual_sha256
        == expected_sha256,
        (
            f'selected_models.json object '
            f'{resolved_path} does not match '
            f'Notebook 52 frozen identity.'
        ),
    )

    referenced_files.append({
        'selected_models_json_path':
            row['json_path'],

        'recorded_value':
            value,

        'relative_path':
            relative_path,

        'path':
            str(resolved_path),

        'suffix':
            resolved_path.suffix.lower(),

        'sha256':
            actual_sha256,

        'notebook52_verified':
            True,
    })


selected_model_files_df = pd.DataFrame(
    referenced_files,
    columns=[
        'selected_models_json_path',
        'recorded_value',
        'relative_path',
        'path',
        'suffix',
        'sha256',
        'notebook52_verified',
    ],
)


selected_model_files_df.to_csv(
    AUDIT
    / 'selected_model_file_resolution.csv',
    index=False,
)


# ------------------------------------------------------------
# 8. Separate serialised estimators/calibrators from records
# ------------------------------------------------------------

joblib_files = (
    selected_model_files_df[
        selected_model_files_df[
            'suffix'
        ].eq('.joblib')
    ].copy()
)


torch_files = (
    selected_model_files_df[
        selected_model_files_df[
            'suffix'
        ].isin(
            ['.pt', '.pth']
        )
    ].copy()
)


other_binary_files = (
    selected_model_files_df[
        ~selected_model_files_df[
            'suffix'
        ].isin(
            [
                '.joblib',
                '.pt',
                '.pth',
            ]
        )
    ].copy()
)


# ------------------------------------------------------------
# 9. Load joblib objects and inspect capabilities
# ------------------------------------------------------------

joblib_rows = []

JOBLIB_OBJECTS = {}


for _, row in joblib_files.iterrows():

    path = Path(
        row['path']
    )

    obj = joblib.load(
        path
    )

    object_key = (
        row[
            'selected_models_json_path'
        ]
    )

    JOBLIB_OBJECTS[
        object_key
    ] = obj

    joblib_rows.append({
        'selected_models_json_path':
            object_key,

        'path':
            str(path),

        'sha256':
            row['sha256'],

        'class_name':
            obj.__class__.__name__,

        'module':
            obj.__class__.__module__,

        'has_predict':
            bool(
                hasattr(
                    obj,
                    'predict',
                )
            ),

        'has_predict_proba':
            bool(
                hasattr(
                    obj,
                    'predict_proba',
                )
            ),

        'has_transform':
            bool(
                hasattr(
                    obj,
                    'transform',
                )
            ),

        'has_classes':
            bool(
                hasattr(
                    obj,
                    'classes_',
                )
            ),

        'has_n_features_in':
            bool(
                hasattr(
                    obj,
                    'n_features_in_',
                )
            ),

        'n_features_in':
            (
                int(
                    obj.n_features_in_
                )
                if hasattr(
                    obj,
                    'n_features_in_',
                )
                else None
            ),
    })


joblib_object_df = pd.DataFrame(
    joblib_rows
)


joblib_object_df.to_csv(
    AUDIT
    / 'selected_joblib_object_capabilities.csv',
    index=False,
)


# ------------------------------------------------------------
# 10. Contract checks that can be made without guessing semantics
# ------------------------------------------------------------

# Notebook 50/52 froze component/predictor records,
# not the V8.2 33x33 CNN channel contract.
require(
    'ordered_predictors' in schema
    or 'predictors' in schema
    or 'ordered_features' in schema,
    (
        'Notebook 50 schema does not contain '
        'the expected ordered predictor contract.'
    ),
)


# Every estimator which advertises n_features_in_ must agree with
# the frozen predictor count.
if len(joblib_object_df):

    estimator_rows = (
        joblib_object_df[
            joblib_object_df[
                'has_predict'
            ]
            | joblib_object_df[
                'has_predict_proba'
            ]
        ]
    )

    for _, object_row in (
        estimator_rows.iterrows()
    ):

        if pd.notna(
            object_row[
                'n_features_in'
            ]
        ):

            require(
                int(
                    object_row[
                        'n_features_in'
                    ]
                )
                == len(PREDICTORS),
                (
                    'Frozen estimator feature-count '
                    'mismatch: '
                    f'{object_row["path"]}; '
                    f'estimator expects '
                    f'{int(object_row["n_features_in"])}, '
                    f'schema contains '
                    f'{len(PREDICTORS)}.'
                ),
            )


# ------------------------------------------------------------
# 11. Preserve the complete interpreted control identity
# ------------------------------------------------------------

control_record = {
    'policy_id':
        policy.get('policy_id'),

    'compatibility_id':
        policy.get(
            'compatibility_id'
        ),

    'notebook52_run':
        str(N52),

    'authoritative_notebook50_run':
        str(
            SOURCE_NOTEBOOK50_RUN
        ),

    'selected_models_path':
        str(
            SELECTED_MODELS_PATH
        ),

    'selected_models_sha256':
        sha256_file(
            SELECTED_MODELS_PATH
        ),

    'predictor_schema_path':
        str(
            PREDICTOR_SCHEMA_PATH
        ),

    'predictor_schema_sha256':
        sha256_file(
            PREDICTOR_SCHEMA_PATH
        ),

    'reason_code_schema_path':
        str(
            REASON_CODE_SCHEMA_PATH
        ),

    'reason_code_schema_sha256':
        sha256_file(
            REASON_CODE_SCHEMA_PATH
        ),

    'predictor_count':
        len(PREDICTORS),

    'ordered_predictors':
        PREDICTORS,

    'selected_file_count':
        int(
            len(
                selected_model_files_df
            )
        ),

    'selected_joblib_count':
        int(
            len(
                joblib_files
            )
        ),

    'selected_torch_count':
        int(
            len(
                torch_files
            )
        ),

    'selected_other_binary_count':
        int(
            len(
                other_binary_files
            )
        ),

    'semantic_interpretation_frozen':
        False,
}


atomic_json(
    AUDIT
    / 'notebook50_selected_model_control_record.json',
    control_record,
)


# ------------------------------------------------------------
# 12. Diagnostic display
# ------------------------------------------------------------

print(
    'Frozen predictor count:',
    len(PREDICTORS),
)

print(
    'Files explicitly referenced by selected_models.json:',
    len(selected_model_files_df),
)

print(
    'Joblib objects:',
    len(joblib_files),
)

print(
    'Torch checkpoints:',
    len(torch_files),
)

print(
    'Other binary objects:',
    len(other_binary_files),
)

print(
    '\nIMPORTANT: this cell has resolved and verified the '
    'Notebook 50 selected objects, but has not guessed which '
    'object means SAFE_REMOVE versus SAFE_PROTECT.'
)

display(
    control_identity_df
)

display(
    selected_models_fields
)

display(
    selected_model_files_df
)

if len(joblib_object_df):
    display(
        joblib_object_df
    )

## 5. Canonical V7.8 page preparation and observable channels

In [ ]:
def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]
    require(np.array_equal(out[mask==0],original[mask==0]),'Outside-mask immutability failure.')
    return out

def prepare_mask(seed,dilation=MASK_DILATION):
    m=np.where(seed>0,255,0).astype(np.uint8)
    if dilation>0: m=cv2.dilate(m,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
    return m

def padded_components(mask):
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8); h,w=mask.shape; boxes=[]
    for i in range(1,n):
        x,y,bw,bh,a=stats[i]
        if a>=8: boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop(); rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):
                    ax0=min(ax0,bx0); ay0=min(ay0,by0); ax1=max(ax1,bx1); ay1=max(ay1,by1); changed=True
                else: rest.append((bx0,by0,bx1,by1))
            boxes=rest; merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def telea_reconstruct(observed,mask):
    result=observed.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        im=observed[y0:y1,x0:x1]; mk=mask[y0:y1,x0:x1]
        cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA); local=result[y0:y1,x0:x1]; local[mk>0]=cand[mk>0]
    return immutable(observed,result,mask)

def background_surface(observed,mask,sigma=45.0):
    visible=(mask==0)&(observed>=200); values=observed.astype(np.float32)*visible; weights=visible.astype(np.float32)
    num=cv2.GaussianBlur(values,(0,0),sigma); den=cv2.GaussianBlur(weights,(0,0),sigma); surface=num/np.maximum(den,1e-4); fallback=cv2.GaussianBlur(observed,(0,0),sigma)
    return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)

def support_maps(observed,mask,band=4):
    dark=(observed<190)&(mask==0)
    h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
    n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8); c=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,hg,a=stats[i]
        if 3<=a<=650 and w<=40 and hg<=40: c[lab==i]=True
    k=np.ones((2*band+1,2*band+1),np.uint8)
    return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(c.astype(np.uint8),k)>0

def geometry_t80(observed,mask,telea):
    paper=background_surface(observed,mask); norm=cv2.divide(observed,np.maximum(paper,1),scale=255); horizontal,vertical,compact=support_maps(observed,mask); m=mask>0
    preserve=m&(horizontal|vertical|compact)&(observed<200); remove=m&~preserve&(observed>=80)&(norm>=80); ambiguous=m&~preserve&~remove
    result=observed.copy(); result[remove]=paper[remove]; result[ambiguous]=telea[ambiguous]; result[preserve]=observed[preserve]
    return immutable(observed,result,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact,'paper':paper,'norm':norm}

def page_channels(observed,mask,routes):
    arrays={
        'obs':observed.astype(np.float32)/255.0,
        'norm':routes['norm'].astype(np.float32)/255.0,
        'mask':(mask>0).astype(np.float32),
        'depth':cv2.distanceTransform((mask>0).astype(np.uint8),cv2.DIST_L2,5).astype(np.float32),
        'horizontal':routes['horizontal'].astype(np.float32),'vertical':routes['vertical'].astype(np.float32),'compact':routes['compact'].astype(np.float32),
        'preserve':routes['preserve'].astype(np.float32),'remove':routes['remove'].astype(np.float32),'ambiguous':routes['ambiguous'].astype(np.float32),
        'edge':(cv2.Canny(observed,70,170)>0).astype(np.float32),
    }
    aliases={'observed':'obs','normalised':'norm','normalized':'norm','authoritative_mask':'mask','mask_distance':'depth','edges':'edge'}
    ordered=[]
    for name in channels:
        key=aliases.get(name,name)
        require(key in arrays,f'Frozen channel {name} is not provided by the canonical builder.')
        ordered.append((name,arrays[key]))
    return ordered

## 6. Frozen CNN model and calibrated inference

In [ ]:
class TopologyCNN(nn.Module):
    def __init__(self, number_of_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(number_of_channels, 24, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(24),
            nn.MaxPool2d(2),

            nn.Conv2d(24, 48, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(48),
            nn.MaxPool2d(2),

            nn.Conv2d(48, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(64, 2)

    def forward(self, value):
        return self.head(self.net(value).flatten(1))

def extract_state_dict(loaded):
    if not isinstance(loaded, dict):
        return loaded

    for key in [
        'state_dict',
        'model_state_dict',
        'remove_model_state_dict',
        'safe_remove_model_state_dict',
    ]:
        if key in loaded:
            return loaded[key]

    # A plain PyTorch state_dict is itself a dictionary of tensors.
    if loaded and all(torch.is_tensor(value) for value in loaded.values()):
        return loaded

    raise RuntimeError(
        'Checkpoint dictionary does not contain a recognised state_dict.'
    )

def load_frozen_model(path):
    path = Path(path)
    require(
        path.suffix.lower() in {'.pt', '.pth'},
        f'Unsupported frozen model type for CNN inference: {path}'
    )

    loaded = torch.load(path, map_location=DEVICE)
    state = extract_state_dict(loaded)
    state = {
        str(key).removeprefix('module.'): value
        for key, value in state.items()
    }

    model = TopologyCNN(len(channels)).to(DEVICE)
    model.load_state_dict(state, strict=True)
    model.eval()
    return model

models = [load_frozen_model(path) for path in MODEL_PATHS]
calibrators = [joblib.load(path) for path in CALIBRATOR_PATHS]

for path, calibrator in zip(CALIBRATOR_PATHS, calibrators):
    require(
        hasattr(calibrator, 'predict'),
        f'Frozen calibrator has no predict method: {path}'
    )

def score_coordinates(channel_arrays, ys, xs):
    ys = np.asarray(ys, dtype=np.int32)
    xs = np.asarray(xs, dtype=np.int32)

    if len(ys) == 0:
        return (
            np.empty(0, dtype=np.float64),
            np.empty((len(models), 0), dtype=np.float64),
        )

    padded_arrays = [
        np.pad(array.astype(np.float32), HALF, mode='reflect')
        for _, array in channel_arrays
    ]

    member_probabilities = []

    for model, calibrator in zip(models, calibrators):
        batches = []

        with torch.no_grad():
            for start in range(0, len(ys), BATCH_SIZE):
                batch_ys = ys[start:start + BATCH_SIZE]
                batch_xs = xs[start:start + BATCH_SIZE]

                tensors = np.stack([
                    np.stack([
                        padded[y:y + PATCH, x:x + PATCH]
                        for padded in padded_arrays
                    ])
                    for y, x in zip(batch_ys, batch_xs)
                ]).astype(np.float32)

                logits = model(torch.from_numpy(tensors).to(DEVICE))
                raw_probability = (
                    torch.softmax(logits, dim=1)[:, 1]
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float64)
                )

                calibrated = np.asarray(
                    calibrator.predict(raw_probability),
                    dtype=np.float64,
                )
                require(
                    np.isfinite(calibrated).all(),
                    'Calibrator emitted non-finite probabilities.'
                )
                batches.append(calibrated)

        member_probabilities.append(np.concatenate(batches))

    member_matrix = np.vstack(member_probabilities)

    if len(models) == 1:
        aggregate = member_matrix[0]
    else:
        require(
            MODEL_AGGREGATION in {
                'mean_calibrated_probability',
                'mean_probability',
            },
            f'Unsupported frozen aggregation: {MODEL_AGGREGATION}'
        )
        aggregate = member_matrix.mean(axis=0)

    require(
        np.isfinite(aggregate).all(),
        'Aggregate SAFE_REMOVE probability is non-finite.'
    )
    require(
        ((aggregate >= 0.0) & (aggregate <= 1.0)).all(),
        'Aggregate SAFE_REMOVE probability is outside [0, 1].'
    )

    return aggregate, member_matrix

def decisions_from_probability(probability):
    probability = np.asarray(probability, dtype=np.float64)
    complement = 1.0 - probability

    decision = np.full(len(probability), 'UNCERTAIN', dtype=object)

    # This label is retained only as an audit category. It does not create
    # SAFE_PROTECT or any other intervention.
    decision[complement >= ANNOTATION_THRESHOLD] = 'FALLBACK_M3'

    safe_remove = (
        (probability >= PRINT_THRESHOLD)
        & ((probability - complement) >= PRINT_MARGIN)
    )
    decision[safe_remove] = 'SAFE_REMOVE'

    return decision

def enforce_safe_remove_run_rule(shape, ys, xs, decisions, probabilities):
    final = np.asarray(decisions, dtype=object).copy()
    candidate_map = np.zeros(shape, dtype=np.uint8)

    candidate_indices = np.where(final == 'SAFE_REMOVE')[0]
    if len(candidate_indices) == 0:
        return final

    candidate_map[
        ys[candidate_indices],
        xs[candidate_indices]
    ] = 1

    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        candidate_map,
        8,
    )

    coordinate_labels = labels[ys, xs]

    for component_id in range(1, count):
        indices = np.where(
            (coordinate_labels == component_id)
            & (final == 'SAFE_REMOVE')
        )[0]

        if len(indices) == 0:
            continue

        area = int(stats[component_id, cv2.CC_STAT_AREA])
        invalid = (
            area < MIN_PRINT_RUN
            or np.any(probabilities[indices] < PRINT_THRESHOLD)
            or (
                float(np.mean(probabilities[indices]))
                < PRINT_THRESHOLD
            )
        )

        if invalid:
            final[indices] = 'UNCERTAIN'

    return final

print('Frozen inference objects loaded:', len(models), 'model(s)')

## 7. Resolve the Notebook 06 → 07 reviewed cohort

In [ ]:
def read_audits(folder,label):
    records=[]; failures=[]
    for p in sorted(Path(folder).glob('*.json')):
        try:
            d=read_json(p); require(isinstance(d,dict),'root is not object'); d['_audit_path']=str(p); d['_type']=label; records.append(d)
        except Exception as e: failures.append({'type':label,'path':str(p),'error':str(e)})
    return records,failures

reviews,rf=read_audits(REVIEW_AUDIT_DIR,'review'); drafts,df=read_audits(DRAFT_AUDIT_DIR,'draft')
pd.DataFrame(rf+df).to_csv(AUDIT/'audit_parse_failures.csv',index=False)
require(not (rf+df),'Unreadable HITL audit records exist.')

def path_from(record,keys,required=True):
    for k in keys:
        v=record.get(k)
        if v and Path(v).exists(): return Path(v),k
    if required: raise RuntimeError(f"{record.get('_audit_path')}: no existing path for {keys}")
    return None,None

drafts_by_md5={}
for d in drafts:
    h=str(d.get('source_md5','')).lower()
    if h: drafts_by_md5.setdefault(h,[]).append(d)

rows=[]; unresolved=[]
for r in reviews:
    try:
        observed,ok=path_from(r,['aligned_output','aligned_page','observed_output'])
        reviewed,mk=path_from(r,['corrected_mask_output','corrected_mask','mask_output'])
        src,_=path_from(r,['source','source_file'],required=False)
        h=str(r.get('source_md5','')).lower()
        candidates=drafts_by_md5.get(h,[])
        draft=candidates[0] if len(candidates)==1 else None
        draft_mask=confidence=None; draft_audit=None
        if draft:
            draft_mask,_=path_from(draft,['draft_mask','draft_mask_output'],required=False)
            confidence,_=path_from(draft,['confidence_map','confidence_map_output'],required=False)
            draft_audit=draft['_audit_path']
        obs=read_gray(observed); seed=read_gray(reviewed); require(obs.shape==seed.shape,'observed/reviewed shape mismatch')
        page=Path(observed).stem
        rows.append({'page':page,'observed_path':str(observed),'reviewed_mask_path':str(reviewed),'source_path':str(src) if src else None,'source_md5':h or None,'review_audit_path':r['_audit_path'],'draft_audit_path':draft_audit,'draft_mask_path':str(draft_mask) if draft_mask else None,'confidence_map_path':str(confidence) if confidence else None,'height':obs.shape[0],'width':obs.shape[1]})
    except Exception as e: unresolved.append({'review_audit_path':r.get('_audit_path'),'error':str(e)})
roster=pd.DataFrame(rows)
if PAGE_ALLOWLIST is not None: roster=roster[roster.page.isin(set(PAGE_ALLOWLIST))].copy()
require(not roster.empty,'No reviewed HITL pages resolved.')
dup=roster[roster.duplicated('page',keep=False)]
require(dup.empty,f'Duplicate page identities: {dup.page.tolist()}')
roster=roster.sort_values('page').reset_index(drop=True)
roster.to_csv(TABLES/'hitl_reviewed_roster.csv',index=False); pd.DataFrame(unresolved).to_csv(TABLES/'unresolved_review_lineages.csv',index=False)
if REQUIRE_EXACT_151: require(len(roster)==EXPECTED_REVIEWED_PAGES,f'Expected 151 pages, resolved {len(roster)}')
print('Resolved reviewed pages:',len(roster),'reference expectation:',EXPECTED_REVIEWED_PAGES)

## 8. Fresh SAFE_REMOVE batch inference and M5 reconstruction

In [ ]:
page_rows = []
pixel_parts = []
failures = []

for position, roster_row in roster.iterrows():
    page = str(roster_row.page)
    page_directory = PAGES / page
    started = time.time()

    try:
        observed = read_gray(roster_row.observed_path)
        seed = read_gray(roster_row.reviewed_mask_path)

        require(
            observed.shape == seed.shape,
            f'{page}: observed/reviewed-mask shape mismatch.'
        )

        authority = prepare_mask(seed, MASK_DILATION)
        m3 = authority > 0

        telea_m3 = telea_reconstruct(observed, authority)
        baseline, routes = geometry_t80(observed, authority, telea_m3)

        eligible = (
            m3
            & (observed < DARK_THRESHOLD)
            & ~routes['preserve']
        )
        ys, xs = np.where(eligible)

        ordered_channels = page_channels(observed, authority, routes)
        calibrated_probability, member_probabilities = score_coordinates(
            ordered_channels,
            ys,
            xs,
        )

        preliminary = decisions_from_probability(calibrated_probability)
        final = enforce_safe_remove_run_rule(
            observed.shape,
            ys,
            xs,
            preliminary,
            calibrated_probability,
        )

        safe_remove = np.zeros(observed.shape, dtype=bool)
        safe_remove[
            ys[final == 'SAFE_REMOVE'],
            xs[final == 'SAFE_REMOVE'],
        ] = True

        require(
            not np.any(safe_remove & ~m3),
            f'{page}: SAFE_REMOVE occurs outside M3.'
        )
        require(
            not np.any(safe_remove & routes['preserve']),
            f'{page}: SAFE_REMOVE overlaps the protected V7.8 route.'
        )

        m5_remove = m3 & ~safe_remove
        require(
            np.array_equal(m5_remove, m3 & ~safe_remove),
            f'{page}: M5_REMOVE formula mismatch.'
        )

        # M5 reconstruction applies the canonical reconstruction only to the
        # remaining M5 mask.
        corrected = telea_reconstruct(
            observed,
            m5_remove.astype(np.uint8) * 255,
        )

        # SAFE_REMOVE is removal/background treatment, never observed-byte
        # preservation. Use the same frozen paper surface already generated
        # by geometry_t80.
        paper = routes['paper']
        corrected[safe_remove] = paper[safe_remove]

        require(
            np.array_equal(corrected[~m3], observed[~m3]),
            f'{page}: outside-authority drift.'
        )
        require(
            np.array_equal(
                corrected[safe_remove],
                paper[safe_remove],
            ),
            f'{page}: SAFE_REMOVE does not equal frozen paper treatment.'
        )

        page_directory.mkdir(parents=True, exist_ok=False)

        cv2.imwrite(str(page_directory / 'observed.png'), observed)
        cv2.imwrite(
            str(page_directory / 'authoritative_seed.png'),
            (seed > 0).astype(np.uint8) * 255,
        )
        cv2.imwrite(
            str(page_directory / 'm3.png'),
            m3.astype(np.uint8) * 255,
        )
        cv2.imwrite(
            str(page_directory / 'safe_remove.png'),
            safe_remove.astype(np.uint8) * 255,
        )
        cv2.imwrite(
            str(page_directory / 'm5_remove.png'),
            m5_remove.astype(np.uint8) * 255,
        )
        cv2.imwrite(
            str(page_directory / 'v7_8_baseline.png'),
            baseline,
        )
        cv2.imwrite(
            str(page_directory / 'safe_remove_paper.png'),
            paper,
        )
        cv2.imwrite(
            str(page_directory / 'corrected.png'),
            corrected,
        )

        probability_image = np.zeros(observed.shape, dtype=np.float32)
        probability_image[ys, xs] = calibrated_probability

        decision_map = np.zeros(observed.shape, dtype=np.uint8)
        decision_map[
            ys[final == 'FALLBACK_M3'],
            xs[final == 'FALLBACK_M3'],
        ] = 85
        decision_map[
            ys[final == 'UNCERTAIN'],
            xs[final == 'UNCERTAIN'],
        ] = 170
        decision_map[
            ys[final == 'SAFE_REMOVE'],
            xs[final == 'SAFE_REMOVE'],
        ] = 255

        np.savez_compressed(
            page_directory / 'fresh_inference.npz',
            eligible=eligible.astype(np.uint8),
            p_safe_remove=probability_image,
            decision_map=decision_map,
            m3=m3.astype(np.uint8),
            safe_remove=safe_remove.astype(np.uint8),
            m5_remove=m5_remove.astype(np.uint8),
            paper=paper,
        )

        cv2.imwrite(
            str(page_directory / 'decision_map.png'),
            decision_map,
        )

        difference = np.zeros_like(observed)
        difference[corrected != baseline] = 255
        cv2.imwrite(
            str(page_directory / 'difference_from_v78.png'),
            difference,
        )

        prediction_rows = pd.DataFrame({
            'page': page,
            'y': ys,
            'x': xs,
            'p_safe_remove': calibrated_probability,
            'decision_pre_run': preliminary,
            'decision': final,
        })

        for model_index in range(member_probabilities.shape[0]):
            prediction_rows[
                f'member_{model_index + 1}_p_safe_remove'
            ] = member_probabilities[model_index]

        part_path = TABLES / f'{page}_fresh_safe_remove.parquet'
        prediction_rows.to_parquet(part_path, index=False)
        pixel_parts.append(part_path)

        result_row = {
            'page': page,
            'status': 'PASS',
            'eligible_pixels': int(len(prediction_rows)),
            'safe_remove_pixels': int(safe_remove.sum()),
            'fallback_m3_pixels': int(
                np.sum(final == 'FALLBACK_M3')
            ),
            'uncertain_pixels': int(np.sum(final == 'UNCERTAIN')),
            'm3_pixels': int(m3.sum()),
            'm5_pixels': int(m5_remove.sum()),
            'outside_authority_drift': int(
                np.sum(corrected[~m3] != observed[~m3])
            ),
            'm5_formula_xor': int(
                np.logical_xor(
                    m5_remove,
                    m3 & ~safe_remove,
                ).sum()
            ),
            'safe_remove_paper_mismatch': int(
                np.sum(
                    corrected[safe_remove]
                    != paper[safe_remove]
                )
            ),
            'seconds': float(time.time() - started),
            'observed_sha256': array_hash(observed),
            'm3_sha256': array_hash(m3.astype(np.uint8)),
            'safe_remove_sha256': array_hash(
                safe_remove.astype(np.uint8)
            ),
            'm5_sha256': array_hash(m5_remove.astype(np.uint8)),
            'paper_sha256': array_hash(paper),
            'corrected_sha256': array_hash(corrected),
        }

        atomic_json(
            page_directory / 'page_manifest.json',
            result_row,
        )
        page_rows.append(result_row)

        print(
            f'[{position + 1}/{len(roster)}] PASS {page}: '
            f'SAFE_REMOVE={result_row["safe_remove_pixels"]:,}'
        )

    except Exception as error:
        failure = {
            'page': page,
            'status': 'FAIL',
            'error': str(error),
            'traceback': traceback.format_exc(),
            'review_audit_path': roster_row.review_audit_path,
        }
        atomic_json(FAILURES / f'{page}.json', failure)
        failures.append(failure)

        print(
            f'[{position + 1}/{len(roster)}] FAIL {page}: {error}'
        )

        if STOP_AT_FIRST_PAGE_FAILURE:
            raise

page_df = pd.DataFrame(page_rows)
fail_df = pd.DataFrame(failures)

page_df.to_csv(TABLES / 'page_results.csv', index=False)
fail_df.to_csv(TABLES / 'page_failures.csv', index=False)

if pixel_parts:
    combined_predictions = pd.concat(
        [pd.read_parquet(path) for path in pixel_parts],
        ignore_index=True,
    )
    combined_predictions.to_parquet(
        TABLES / 'all_fresh_safe_remove_predictions.parquet',
        index=False,
    )

print('Passed:', len(page_df), 'Failed:', len(fail_df))

## 9. Deterministic replay and operational integrity verdict

In [ ]:
require(len(page_df) > 0, 'No page completed successfully.')

sample_page = str(page_df.iloc[0].page)
sample_roster = roster.set_index('page').loc[sample_page]

sample_observed = read_gray(sample_roster.observed_path)
sample_m3 = read_gray(PAGES / sample_page / 'm3.png') > 0
sample_safe_remove = (
    read_gray(PAGES / sample_page / 'safe_remove.png') > 0
)
sample_m5 = read_gray(PAGES / sample_page / 'm5_remove.png') > 0
sample_paper = read_gray(
    PAGES / sample_page / 'safe_remove_paper.png'
)
sample_saved = read_gray(PAGES / sample_page / 'corrected.png')

sample_replayed = telea_reconstruct(
    sample_observed,
    sample_m5.astype(np.uint8) * 255,
)
sample_replayed[sample_safe_remove] = (
    sample_paper[sample_safe_remove]
)

replay_record = {
    'page': sample_page,
    'm5_formula_exact': bool(
        np.array_equal(
            sample_m5,
            sample_m3 & ~sample_safe_remove,
        )
    ),
    'safe_remove_paper_exact': bool(
        np.array_equal(
            sample_saved[sample_safe_remove],
            sample_paper[sample_safe_remove],
        )
    ),
    'corrected_exact': bool(
        np.array_equal(sample_saved, sample_replayed)
    ),
    'saved_sha256': array_hash(sample_saved),
    'replayed_sha256': array_hash(sample_replayed),
}

atomic_json(AUDIT / 'deterministic_replay.json', replay_record)

checks = {
    'notebook52_verdict_passed': (
        v52.get('verdict')
        == 'PASS_M5_REMOVE_FREEZE_READY_FOR_OPERATIONAL_BUILDER'
    ),
    'frozen_artifact_hashes_passed': True,
    'fresh_inference_contract_unambiguous': True,
    'all_resolved_pages_accounted_for': (
        len(page_df) + len(fail_df) == len(roster)
    ),
    'zero_page_failures': len(fail_df) == 0,
    'zero_outside_authority_drift': bool(
        len(page_df)
        and (page_df.outside_authority_drift == 0).all()
    ),
    'zero_m5_formula_xor': bool(
        len(page_df)
        and (page_df.m5_formula_xor == 0).all()
    ),
    'safe_remove_uses_paper_treatment': bool(
        len(page_df)
        and (page_df.safe_remove_paper_mismatch == 0).all()
    ),
    'deterministic_replay_exact': bool(
        replay_record['m5_formula_exact']
        and replay_record['safe_remove_paper_exact']
        and replay_record['corrected_exact']
    ),
    'no_clean_truth_loaded': True,
    'safe_protect_disabled': True,
    'no_retraining_or_recalibration': True,
}

first_failure = next(
    (name for name, passed in checks.items() if not passed),
    None,
)

verdict = (
    'PASS_V9_0B_FRESH_SAFE_REMOVE_BATCH'
    if first_failure is None
    else 'FAIL_OR_PARTIAL_V9_0B_BATCH'
)

record = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'version': '60R1-v9.0B-fresh-safe-remove-HITL-corrected',
    'verdict': verdict,
    'first_failed_gate': first_failure,
    'checks': checks,
    'resolved_pages': int(len(roster)),
    'passed_pages': int(len(page_df)),
    'failed_pages': int(len(fail_df)),
    'expected_page_reference': EXPECTED_REVIEWED_PAGES,
    'policy_id': policy.get('policy_id'),
    'compatibility_id': policy.get('compatibility_id'),
    'notebook52_run': str(N52),
    'safe_remove_total': (
        int(page_df.safe_remove_pixels.sum())
        if len(page_df)
        else 0
    ),
    'm3_total': (
        int(page_df.m3_pixels.sum())
        if len(page_df)
        else 0
    ),
    'm5_total': (
        int(page_df.m5_pixels.sum())
        if len(page_df)
        else 0
    ),
    'safe_remove_treatment': 'frozen_background_surface',
    'clean_truth_used': False,
    'safe_protect_enabled': False,
    'automatic_production_authority': False,
}

atomic_json(OUT / 'notebook60r1_verdict.json', record)
print(json.dumps(record, indent=2))

if first_failure is not None and STOP_AT_FIRST_PAGE_FAILURE:
    raise RuntimeError(
        f'Notebook 60R1 failed at gate: {first_failure}'
    )

## 10. Visual review queue

In [ ]:
queue=page_df.copy(); queue['safe_remove_fraction']=queue.safe_remove_pixels/np.maximum(queue.m3_pixels,1); queue=queue.sort_values(['safe_remove_fraction','safe_remove_pixels'],ascending=False); queue.to_csv(TABLES/'visual_review_queue.csv',index=False)
for rank,(_,r) in enumerate(queue.head(MAX_VISUAL_PAGES).iterrows(),1):
    page=r.page; observed=read_gray(PAGES/page/'observed.png'); m3=read_gray(PAGES/page/'m3.png')>0; safe=read_gray(PAGES/page/'safe_remove.png')>0; m5=read_gray(PAGES/page/'m5_remove.png')>0; base=read_gray(PAGES/page/'v7_8_baseline.png'); corrected=read_gray(PAGES/page/'corrected.png')
    overlay=cv2.cvtColor(observed,cv2.COLOR_GRAY2RGB); overlay[m3]=(255,180,0); overlay[safe]=(0,200,0); overlay[m5]=(220,80,40)
    fig,ax=plt.subplots(1,4,figsize=(20,7)); ims=[observed,overlay,base,corrected]; titles=['Observed','M3 orange / SAFE_REMOVE green / M5 red','V7.8 baseline','V8.8 REMOVE-only result']
    for a,im,t in zip(ax,ims,titles): a.imshow(im,cmap='gray' if im.ndim==2 else None); a.set_title(t); a.axis('off')
    fig.suptitle(f'{page} | SAFE_REMOVE={int(r.safe_remove_pixels):,} | M3={int(r.m3_pixels):,} | fraction={r.safe_remove_fraction:.4f}'); plt.tight_layout(); fig.savefig(VISUAL/f'{rank:03d}_{page}.png',dpi=130,bbox_inches='tight'); plt.close(fig)
print('Visual cases:',min(MAX_VISUAL_PAGES,len(queue)))

## 11. Identity, summary and final upload package

In [ ]:
environment={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'run':str(OUT),'version':'60R1-v9.0B-fresh-safe-remove-HITL','notebook52':str(N52),'policy_sha256':sha256_file(POLICY_PATH),'inventory_sha256':sha256_file(INVENTORY_PATH),'replay_sha256':sha256_file(REPLAY_PATH),'schema_sha256':sha256_file(SCHEMA_PATH),'models':[{'path':str(p),'sha256':sha256_file(p)} for p in MODEL_PATHS],'calibrators':[{'path':str(p),'sha256':sha256_file(p)} for p in CALIBRATOR_PATHS],'contract':contract,'reviewed_roster_sha256':sha256_file(TABLES/'hitl_reviewed_roster.csv'),'page_results_sha256':sha256_file(TABLES/'page_results.csv'),'predictions_sha256':sha256_file(TABLES/'all_fresh_safe_remove_predictions.parquet') if (TABLES/'all_fresh_safe_remove_predictions.parquet').exists() else None,'environment':environment}
atomic_json(OUT/'experiment_identity.json',identity)
lines=['NOTEBOOK 60R1 / V9.0B FRESH SAFE_REMOVE HITL OPERATIONALISATION','',f'Verdict: {verdict}',f'Reviewed pages resolved: {len(roster)}',f'Passed pages: {len(page_df)}',f'Failed pages: {len(fail_df)}',f'Frozen policy ID: {policy.get("policy_id")}',f'SAFE_REMOVE pixels: {int(page_df.safe_remove_pixels.sum()):,}',f'M3 pixels: {int(page_df.m3_pixels.sum()):,}',f'M5_REMOVE pixels: {int(page_df.m5_pixels.sum()):,}','','No Notebook 45/46/50 page-output dependency.','No clean truth at inference.','No retraining, refitting, recalibration or threshold movement.','Production authority remains false.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))

required=[OUT/'executive_summary.txt',OUT/'notebook60r1_verdict.json',OUT/'experiment_identity.json',AUDIT/'notebook52_source.json',AUDIT/'fresh_inference_contract.json',AUDIT/'resolved_frozen_artifacts.csv',AUDIT/'deterministic_replay.json',TABLES/'hitl_reviewed_roster.csv',TABLES/'page_results.csv',TABLES/'page_failures.csv',TABLES/'visual_review_queue.csv']
if (TABLES/'all_fresh_safe_remove_predictions.parquet').exists(): required.append(TABLES/'all_fresh_safe_remove_predictions.parquet')
for p in required: require(p.exists(),f'Missing upload artifact: {p}')
rows=[]
for src in required+sorted(VISUAL.glob('*.png'))[:MAX_VISUAL_PAGES]:
    cat='visuals' if src.suffix.lower()=='.png' else ('tables' if src.parent==TABLES else 'audit' if src.parent==AUDIT else 'root'); dst=UPLOAD/cat/src.name; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(rows).sort_values('relative_path').to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 60R1 fresh SAFE_REMOVE inference on Notebook 07 reviewed HITL masks. Research/shadow evidence only; no automatic production authority.' + chr(10))
zip_path=UPLOAD/'60R1_V9_0B_review_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file() and p!=zip_path: z.write(p,p.relative_to(UPLOAD))
require(zip_path.exists() and zip_path.stat().st_size>0,'Upload bundle creation failed.')
print(chr(10).join(lines)); print('Upload bundle:',zip_path)